# U04 · Modelos supervisados I — lineal, logística y Naïve Bayes

> **Para quien pone el criterio clínico, no el código.** Aquí entrenamos los **tres modelos supervisados más simples y fundamentales**, los que sostienen buena parte de las herramientas de riesgo que ya usas. No los tratamos como calentamiento: a menudo son el **mejor punto de partida** y, más veces de lo que se cree, también el **modelo final**.

> ⚠️ Todos los datos son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** La primera celda los crea sola: no hay que descargar nada.

**Qué haremos, en clave clínica:**
1. **Regresión lineal** sobre `riesgo_cv_10a` (%): predecir un **número** y —lo más valioso— **leer sus coeficientes** ("cada +10 mmHg de tensión sube el riesgo en X puntos").
2. **Regresión logística** sobre `evento_cv` (0/1): devolver una **probabilidad** y traducir sus coeficientes a **odds ratio** —el idioma con el que la medicina comunica el riesgo—.
3. **Naïve Bayes** sobre `notas_clinicas.csv`: clasificar **texto clínico** por especialidad, rapidísimo y explicable.

**Regla de oro:** *empieza siempre por el modelo más simple que pueda funcionar.* En salud vale doble: rápido, **interpretable** y un *baseline* honesto contra el que medir cualquier cosa más compleja.

[Abrir en Colab](https://colab.research.google.com/drive/1mpLBIam_ebKCo7P6PpXDJHnkJBrbhDRz)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno del curso **genera los ficheros sintéticos** en la carpeta de trabajo. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo. Recuerda: son datos **inventados**, sirven para aprender, no para decidir sobre personas.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. Conoce tus datos antes de modelar

Regla sagrada del curso: **primero mira los datos, luego modela.** Abrimos la tabla central `pacientes.csv` (20 000 pacientes sintéticos) y nos asomamos a sus primeras filas para ver qué pinta tiene.


In [ ]:
import pandas as pd
import numpy as np

pacientes = pd.read_csv("pacientes.csv")
print("Filas y columnas:", pacientes.shape)
pacientes.head()

**Lo que vemos:** cada **fila** es un paciente; cada **columna**, un dato clínico suyo. Están las variables de siempre —edad, sexo, IMC, tensión (`ta_sistolica`/`ta_diastolica`), glucemia, colesterol, HDL, hábitos— y **dos columnas objetivo** que usaremos por separado:

- `riesgo_cv_10a` → un **porcentaje** de riesgo cardiovascular a 10 años. Es un **número** → problema de **regresión** (bloque 1).
- `evento_cv` → **0/1**, si ocurre o no un evento. Es una **etiqueta** → problema de **clasificación** (bloque 2).


Miremos dos cosas clave antes de empezar: cómo se reparte el **riesgo** (el número que predeciremos) y qué **prevalencia** tiene el evento (cuántos 1 hay). Eso nos dice contra qué "listón" competimos.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(pacientes["riesgo_cv_10a"], bins=30, color="#00AEC7", edgecolor="white")
ax[0].set_xlabel("Riesgo cardiovascular a 10 años (%)")
ax[0].set_ylabel("Nº de pacientes")
ax[0].set_title("Distribución del riesgo (regresión)")

prev = pacientes["evento_cv"].mean()
ax[1].bar(["Sin evento (0)", "Con evento (1)"],
          [(pacientes["evento_cv"] == 0).mean(), prev],
          color=["#9AA7B0", "#E4572E"], edgecolor="white")
ax[1].set_ylabel("Proporción de pacientes")
ax[1].set_title(f"Prevalencia de evento_cv = {prev:.1%}")

plt.tight_layout(); plt.show()
print(f"Prevalencia de evento cardiovascular: {prev:.1%}")

**Lo que vemos:** el riesgo se concentra en valores bajos-medios con una cola hacia riesgos altos (como una población real). Y el evento es **poco frecuente**: unos **19 %** de casos positivos. Este dato importa: un modelo tonto que dijera "nadie tendrá evento" acertaría el ~81 % de las veces. Por eso la **exactitud a secas engaña** con clases desbalanceadas y, en clasificación, miraremos el **AUC** (U3) en lugar de solo el porcentaje de aciertos.


## 2. Regresión lineal: predecir el riesgo (y leer sus coeficientes)

La regresión lineal es el modelo más simple que existe: busca la mejor **combinación ponderada** de las variables para estimar un número. Cada variable aporta un **peso (coeficiente)** con **signo y magnitud** legibles. Vamos a estimar `riesgo_cv_10a` y, sobre todo, a **leer esos pesos en clave clínica**.


### 2.1 Preparamos las variables

Las variables **numéricas** (edad, tensión…) las usamos tal cual. Las **categóricas** (`sexo`, `tabaquismo`, `actividad_fisica`) hay que convertirlas en columnas 0/1 (*one-hot*), eligiendo un **nivel de referencia sano**: `nunca` para tabaquismo, `alta` para actividad, `F` para sexo. Así cada coeficiente se leerá como *"cuánto sube o baja el riesgo respecto a ese nivel sano"*.

**Importante:** aquí **no** normalizamos las variables. Las dejamos en sus **unidades naturales** (mmHg, mg/dL, años) para que un coeficiente signifique *"puntos de riesgo por cada unidad real"* —justo el lenguaje de la consulta.


In [ ]:
from sklearn.model_selection import train_test_split

num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]

# Fijamos el nivel SANO como referencia (se elimina con drop_first)
d = pacientes.copy()
d["sexo"]            = pd.Categorical(d["sexo"],            ["F", "M"])
d["tabaquismo"]      = pd.Categorical(d["tabaquismo"],      ["nunca", "ex", "activo"])
d["actividad_fisica"] = pd.Categorical(d["actividad_fisica"], ["alta", "media", "baja"])
cat = ["sexo", "tabaquismo", "actividad_fisica"]

X = pd.get_dummies(d[num + cat], columns=cat, drop_first=True)
y_riesgo = pacientes["riesgo_cv_10a"]

# Partición honesta: 75% para entrenar, 25% para evaluar (nunca visto por el modelo)
Xtr, Xte, ytr, yte = train_test_split(X, y_riesgo, test_size=0.25, random_state=42)
print("Variables que entran al modelo:")
print(list(X.columns))

**Lo que hemos hecho:** cada categoría se ha vuelto una columna 0/1 (p. ej. `tabaquismo_activo`, `tabaquismo_ex`; la ausencia de ambas = "nunca", el nivel de referencia). Y hemos apartado un **25 % de test** que el modelo **no verá al entrenar**, para medir de forma honesta.


### 2.2 Entrenamos y medimos

Entrenar una regresión lineal es una línea. Después la evaluamos en el test con dos métricas de U3:
- **MAE** (error absoluto medio): *"de media, me equivoco en X puntos de riesgo"*. Cuanto menor, mejor.
- **R²**: qué fracción de la variabilidad del riesgo explica el modelo (1 = perfecto, 0 = no explica nada).


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

lineal = LinearRegression().fit(Xtr, ytr)
pred = lineal.predict(Xte)

print("R2 :", round(r2_score(yte, pred), 3))
print("MAE:", round(mean_absolute_error(yte, pred), 2), "puntos de riesgo")

**Lo que significa:** un **R² ≈ 0,81** quiere decir que, con solo una suma ponderada de variables, el modelo ya explica el **81 %** de la variación del riesgo. Un **MAE ≈ 7 puntos** dice que, de media, su estimación se desvía unos 7 puntos porcentuales del riesgo real de cada paciente sintético. Para ser el modelo más simple posible, es un *baseline* **muy digno**.


### 2.3 El corazón de la regresión lineal: los coeficientes

Aquí está su gran regalo para un clínico. Cada variable tiene un **peso** que dice **cuánto y en qué sentido** mueve el riesgo, en **puntos porcentuales por unidad**. Los ordenamos de mayor a menor.


In [ ]:
coefs = pd.Series(lineal.coef_, index=X.columns).sort_values(ascending=False)
coefs.round(3)

**Cómo se leen (todo "manteniendo lo demás constante"):**

- **`diabetes` ≈ +12 puntos** y **`tabaquismo_activo` ≈ +10 puntos**: ser diabético, o fumar de forma activa **frente a no fumar**, dispara el riesgo estimado. Son las palancas más potentes.
- **`sexo_M` ≈ +5,8** y **`antecedentes_familiares` ≈ +5**: ser hombre o tener antecedentes suma varios puntos.
- **`edad` ≈ +0,74 por año** e **`imc` ≈ +0,58 por punto de IMC**: efectos por unidad más pequeños, pero que **se acumulan** (20 años de más ≈ +15 puntos).
- **`hdl` ≈ −0,31**: el **único signo negativo claro** → es **protector**, como esperábamos del "colesterol bueno". Cada +1 mg/dL de HDL baja el riesgo estimado.

> Un apunte honesto: la **actividad física** aparece con coeficiente casi nulo aquí. No es que no importe —es que su efecto viaja sobre todo **de forma indirecta** (mejora el IMC y el HDL), y esos ya están en el modelo. La regresión lineal reparte el crédito entre variables correlacionadas; es un límite a tener presente.


### 2.4 La frase que un clínico quiere oír

Traduzcamos un par de coeficientes al lenguaje de la consulta: *"¿cuánto sube el riesgo por cada +10 mmHg de tensión?"*


In [ ]:
for var, delta, unidad in [("ta_sistolica", 10, "+10 mmHg de tensión sistólica"),
                            ("edad", 10, "+10 años de edad"),
                            ("imc", 5, "+5 puntos de IMC"),
                            ("hdl", 10, "+10 mg/dL de HDL")]:
    efecto = lineal.coef_[list(X.columns).index(var)] * delta
    signo = "sube" if efecto > 0 else "baja"
    print(f"{unidad:35s} -> el riesgo {signo} {abs(efecto):.1f} puntos")

**Esto es oro clínico:** el modelo afirma, por ejemplo, que **cada +10 mmHg de tensión sistólica sube el riesgo estimado ~2 puntos**, y que **cada +10 mg/dL de HDL lo baja ~3 puntos**. Es exactamente el tipo de afirmación transparente y auditable con la que se razona el riesgo en una consulta o en un comité —a veces **más útil** que ganar unas décimas con un modelo opaco.


### 2.5 ¿Acierta? Predicho frente a real

Una gráfica lo cuenta mejor que un número: dibujamos el riesgo **predicho** (eje X) frente al **real** (eje Y) para los pacientes de test. Si el modelo fuera perfecto, todos los puntos caerían sobre la **diagonal**.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(pred, yte, s=8, alpha=0.3, color="#00AEC7")
lims = [0, max(pred.max(), yte.max())]
plt.plot(lims, lims, "--", color="#E4572E", label="Predicción perfecta")
plt.xlabel("Riesgo predicho por el modelo (%)")
plt.ylabel("Riesgo real del paciente (%)")
plt.title("Regresión lineal: predicho vs. real (test)")
plt.legend(); plt.tight_layout(); plt.show()

**Lo que vemos:** la nube sigue claramente la diagonal —el modelo capta la tendencia—, pero con **dispersión**: se desvía varios puntos en cada paciente (ese es el MAE≈7 que calculamos). En los riesgos altos tiende a quedarse algo corto: señal de que hay **relaciones no lineales e interacciones** (la tensión no pesa igual en un diabético) que una recta no puede ver.


### 2.6 ¿Compensa un modelo más complejo? (probar y medir)

La regla de oro no dice "usa siempre lo simple", dice "empieza por lo simple y **demuestra** si lo complejo mejora". Comparémoslo con un **Random Forest**, un modelo mucho más sofisticado que **sí capta interacciones**.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
r2_rf = r2_score(yte, rf_reg.predict(Xte))
print("R2 regresión lineal :", round(r2_score(yte, pred), 3))
print("R2 Random Forest    :", round(r2_rf, 3))

**La lección:** aquí **sí gana el modelo complejo** (R² ≈ 0,91 frente a 0,81), justamente porque **captura interacciones** que la recta no ve. Pero fíjate: hemos tenido que **medirlo**, no presuponerlo. Y aun perdiendo en precisión, la lineal conserva algo que el bosque **no da**: coeficientes legibles. Interpretabilidad primero; potencia, solo si compensa —y demostrado.


## 3. Regresión logística: el caballo de batalla de la epidemiología

Si hay un modelo que un profesional sanitario debe conocer al dedillo, es este. A pesar del nombre, **clasifica**: devuelve la **probabilidad** de que ocurra `evento_cv`. Y su regalo es que sus coeficientes, al exponenciarse, se leen como **odds ratio** —el idioma con el que la medicina comunica el riesgo—.


### 3.1 Preparamos y entrenamos

Reutilizamos las mismas variables `X`, pero ahora el objetivo es `evento_cv` (0/1). Partimos **estratificando** (para que train y test tengan la misma proporción de eventos, ~19 %). No regularizamos el modelo, para que los coeficientes sean **directamente interpretables** como odds ratio.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_evento = pacientes["evento_cv"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y_evento, test_size=0.25, random_state=42, stratify=y_evento)

logistica = LogisticRegression(max_iter=5000, penalty=None).fit(Xtr, ytr)
prob = logistica.predict_proba(Xte)[:, 1]   # probabilidad de evento (clase 1)
print("AUC:", round(roc_auc_score(yte, prob), 3))

**Lo que significa:** un **AUC ≈ 0,84** indica una capacidad **buena** para ordenar pacientes por riesgo: dado un paciente con evento y otro sin él, el modelo asigna mayor probabilidad al primero el ~84 % de las veces. Nada mal para un modelo lineal.


### 3.2 Lo que la logística hace y la lineal no: dar una probabilidad

No dice solo "sí/no", dice *"este paciente tiene un 12 % de probabilidad de evento"*. Miremos la probabilidad estimada para unos cuantos pacientes de test, de menos a más riesgo.


In [ ]:
muestra = pd.DataFrame({"prob_evento": prob}, index=Xte.index)
muestra["riesgo_%"] = (muestra["prob_evento"] * 100).round(1)
muestra = muestra.sort_values("prob_evento")
# Un paciente de bajo, uno de medio y uno de alto riesgo estimado
muestra.iloc[[0, len(muestra)//2, -1]][["riesgo_%"]]

**Esa cifra es oro clínico.** Permite **graduar** decisiones en vez de un corte rígido: por defecto asignamos "evento" si la probabilidad supera 0,5, pero ese umbral es **negociable** según el coste del error (U3). En prevención, donde **no detectar** un caso (falso negativo) suele ser lo más caro, se **baja el umbral** para capturar a más pacientes de riesgo, asumiendo más falsas alarmas que una segunda valoración filtrará.


### 3.3 El corazón de la logística: los odds ratio

En la lineal, un coeficiente se leía "en puntos". En la logística, **exp(coeficiente) = odds ratio (OR)**: en cuánto se **multiplican las *odds*** del evento cuando la variable sube una unidad (o pasa de "ausente" a "presente"). **OR = 1** → no cambia; **> 1** → aumenta el riesgo; **< 1** → protege.


In [ ]:
odds_ratio = pd.Series(np.exp(logistica.coef_[0]), index=X.columns).sort_values(ascending=False)
odds_ratio.round(3)

**Cómo se leen estos odds ratio (el idioma del riesgo clínico):**

- **`tabaquismo_activo` OR ≈ 2,7**: fumar de forma activa **multiplica por ~2,7 las *odds*** de evento **frente a no fumar**. Casi las triplica. (El exfumador, OR ≈ 1,3: riesgo intermedio, coherente con la clínica.)
- **`sexo_M` ≈ 1,78**, **`diabetes` ≈ 1,72**, **`antecedentes_familiares` ≈ 1,70**: cada uno multiplica las *odds* por ~1,7 → factores de riesgo claros.
- **`hdl` ≈ 0,97 por mg/dL** y **`actividad_fisica_media` ≈ 0,89**: **OR < 1** → **protectores** (empujan el riesgo hacia abajo).

Este es, literalmente, el idioma con el que la medicina comunica el riesgo: *"el doble de probabilidad de…"*, "factor protector". Que el modelo lo hable **de fábrica**, sin cajas negras, es la razón de que la logística sea el **caballo de batalla** de cohortes, casos-control y *scores* como **Framingham** o **SCORE**.


### 3.4 Odds ratio por incrementos clínicamente relevantes

Para variables continuas, el OR "por 1 unidad" (p. ej. +1 mg/dL de HDL) queda cerca de 1 y despista. Es más útil el OR por un **incremento con sentido clínico**: +10 mmHg, +10 años, +10 mg/dL.


In [ ]:
for var, delta, texto in [("tabaquismo_activo", 1, "fumar activo (vs no fumar)"),
                           ("edad", 10, "+10 años de edad"),
                           ("ta_sistolica", 10, "+10 mmHg de tensión sistólica"),
                           ("hdl", 10, "+10 mg/dL de HDL (protector)")]:
    or_delta = np.exp(logistica.coef_[0][list(X.columns).index(var)] * delta)
    efecto = "multiplica" if or_delta > 1 else "divide"
    factor = or_delta if or_delta > 1 else 1 / or_delta
    print(f"{texto:32s} -> OR={or_delta:.2f}  ({efecto} las odds por {factor:.2f})")

**En lenguaje clínico:** **+10 años** casi **duplican** las *odds* de evento (OR ≈ 1,9); **+10 mmHg** las suben ~20 % (OR ≈ 1,2); y **+10 mg/dL de HDL** las **dividen** por ~1,3 (OR ≈ 0,78) → confirmando su papel **protector**. Justo el tipo de frase que aparece en un artículo de epidemiología.


### 3.5 La gran lección: aquí lo simple gana a lo complejo

Comparemos la logística con un **Random Forest** clasificando el mismo `evento_cv`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
auc_rf = roc_auc_score(yte, rf_clf.predict_proba(Xte)[:, 1])
print("AUC regresión logística:", round(roc_auc_score(yte, prob), 3))
print("AUC Random Forest       :", round(auc_rf, 3))

**Y aquí —al revés que en la regresión— gana lo simple:** la logística (AUC ≈ 0,84) **supera** al Random Forest (≈ 0,83), un modelo mucho más sofisticado. ¿Por qué? Porque en estos datos el riesgo es aproximadamente **log-aditivo**: los factores se **acumulan** justo en la escala que la logística modela de forma natural. Cuando la **estructura del problema encaja con un modelo simple**, este es más preciso, más barato y —crucial en salud— **más explicable**.

> Es la ilustración perfecta de la regla de oro: **no siempre gana lo complejo**. Contrasta con el bloque 1, donde para la *regresión* del riesgo sí ganaba el bosque por las interacciones. Moraleja: **probar y medir**, nunca presuponer.


## 4. Naïve Bayes: clasificar texto clínico

En salud el **texto** está por todas partes: notas de evolución, motivos de consulta, informes. Naïve Bayes es un clasificador probabilístico **rapidísimo** basado en el teorema de Bayes, ideal para texto. Lo usaremos para adivinar la **especialidad** de una nota a partir únicamente de sus **palabras**.


### 4.1 Conoce el texto

Abrimos `notas_clinicas.csv` (sintético): una columna `texto` con la nota y una columna `especialidad` con la respuesta correcta. Miremos un par de ejemplos y cuántas notas hay por especialidad.


In [ ]:
notas = pd.read_csv("notas_clinicas.csv")
print("Notas totales:", len(notas))
print("\nNotas por especialidad:")
print(notas["especialidad"].value_counts())
print("\nEjemplos:")
for _, fila in notas.groupby("especialidad").head(1).iterrows():
    print(f"  [{fila['especialidad']:14s}] {fila['texto']}")

**Lo que vemos:** cinco especialidades bien repartidas y notas de texto libre corto. A ojo ya intuimos el patrón: "dolor torácico" suena a **cardiología**; "cefalea" y "focalidad" a **neurología**. Naïve Bayes va a **aprender ese patrón** a partir de las palabras.


### 4.2 Entrenamos (sin fuga de datos)

Dos piezas encadenadas en un *pipeline*:
1. **TfidfVectorizer**: convierte cada texto en números (una columna por palabra, pesando más las palabras informativas y menos las comunes).
2. **MultinomialNB**: el clasificador Naïve Bayes.

Clave anti-fuga: partimos **antes** y ajustamos el vectorizador **solo con el train**. Comparamos con un *baseline* que siempre dice la clase mayoritaria.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

Xtr, Xte, ytr, yte = train_test_split(
    notas["texto"], notas["especialidad"],
    test_size=0.25, random_state=0, stratify=notas["especialidad"])

modelo_texto = make_pipeline(TfidfVectorizer(), MultinomialNB()).fit(Xtr, ytr)

baseline = ytr.value_counts(normalize=True).max()   # acertar siempre la clase más común
print("Exactitud Naïve Bayes:", round(modelo_texto.score(Xte, yte), 3))
print("Exactitud baseline    :", round(baseline, 3), "(decir siempre la especialidad más frecuente)")

**Lo que significa:** el modelo clasifica la especialidad con **altísima exactitud**, muy por encima del *baseline* (~0,20, porque hay 5 clases parejas). Un aviso honesto: aquí acierta **casi perfecto** porque el texto es **sintético y muy separable** (plantillas con palabras muy características). Con **notas reales** —más ruidosas, con abreviaturas y solapamientos— bajaría, pero seguiría siendo un **primer triaje** rápido y utilísimo como *baseline*.


### 4.3 Probémoslo con notas nuevas

Le damos tres notas inventadas por nosotros (que el modelo no ha visto) y vemos qué especialidad predice.


In [ ]:
ejemplos = [
    "dolor torácico opresivo irradiado a brazo izquierdo con sensación de disnea",
    "cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
    "caída casual con dolor e impotencia funcional y deformidad en la muñeca",
]
for texto, pred in zip(ejemplos, modelo_texto.predict(ejemplos)):
    print(f"[{pred:14s}] <- {texto}")

**Lo que vemos:** clasifica cada nota en la especialidad esperada —cardiología, neurología y traumatología— guiándose solo por palabras clave ("torácico", "cefalea", "muñeca").

**¿Por qué funciona, si su suposición es falsa?** El "ingenuo" de Naïve Bayes es que trata cada palabra como **independiente** de las demás —algo que en texto clínico casi nunca es cierto ("dolor" y "torácico" van juntos)—. Pero para **acertar la clase** no necesita la probabilidad exacta, solo **ordenar** bien las opciones: aunque cuente doble la evidencia de palabras correlacionadas, el "ganador" suele seguir siendo el mismo. Un modelo "incorrecto" en sus supuestos que resulta **muy útil** en la práctica.


{% hint style="warning" %}
**⚠️ Aviso · No confundas "clasificar texto" con "estimar riesgo"**

Naïve Bayes es excelente para **enrutar** una nota (¿a qué especialidad?), pero sus probabilidades están **mal calibradas**: un "0,8" suyo **no** significa "80 % real". Para una **probabilidad fiable** de un desenlace clínico, la herramienta es la **regresión logística** —bien calibrada—, no Naïve Bayes. Cada modelo, en su sitio.
{% endhint %}


## 5. Regresión vs. clasificación: ¿qué enfoque elijo?

La pregunta clave es **qué tipo de salida** necesita la decisión clínica.

| Si la pregunta clínica es...                    | Es un problema de...        | Modelo de partida    |
| ----------------------------------------------- | --------------------------- | -------------------- |
| ¿Qué riesgo cardiovascular a 10 años tiene?     | Regresión                   | Regresión lineal     |
| ¿Este paciente tendrá un evento?                | Clasificación (probabilidad)| Regresión logística  |
| ¿A qué especialidad corresponde esta nota?      | Clasificación multiclase    | Naïve Bayes          |

Y cuando lo que de verdad hace falta es una **probabilidad fiable**, la **logística** suele ser la respuesta.


## 6. Qué hemos aprendido

- **Empieza por lo simple.** Estos tres modelos son rápidos, **interpretables** y un *baseline* honesto; a menudo, el modelo final. Lo complejo hay que **demostrarlo** con métricas (lo vimos: RF ganó en la *regresión*, la logística ganó en la *clasificación*).
- **La regresión lineal se lee por sus coeficientes.** "Cada +10 mmHg sube el riesgo ~2 puntos", "el HDL es protector". Su techo: **asume linealidad** y reparte mal el crédito entre variables correlacionadas.
- **La regresión logística es el caballo de batalla clínico.** Devuelve **probabilidad**, sus coeficientes son **odds ratio** —el idioma del riesgo—, y aquí **igualó o superó** a lo complejo (AUC ≈ 0,84) porque el riesgo es **log-aditivo**.
- **Naïve Bayes clasifica texto rapidísimo.** Ideal para enrutar notas por especialidad; el "ingenuo" casi nunca es cierto, pero funciona. Ojo: sus probabilidades están **mal calibradas** → para riesgo, usa la logística.

**Puente a la U5:** estos tres modelos comparten un techo —sus fronteras son esencialmente **lineales**—. Cuando los datos exigen **interacciones, umbrales y curvas**, entran los **árboles**, las **SVM** y, sobre todo, los ***ensembles*** (Random Forest, *boosting*): los reyes del dato tabular, que veremos en la Unidad 5.


## 7. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio, la IA escribe el código.** Un buen encargo para reproducir este cuaderno sería:

> *"Con `pacientes.csv` (datos sintéticos), en español y por celdas: (1) entrena una **regresión lineal** para `riesgo_cv_10a` sin escalar las variables, reporta R² y MAE en puntos y **muéstrame los coeficientes** interpretados en clave clínica (efecto de +10 mmHg de tensión, y comprueba que el HDL sale protector). (2) Entrena una **regresión logística** para `evento_cv` (estratificado, sin fuga), dame el **AUC** y convierte los coeficientes en **odds ratio** (exp del coeficiente) explicando el del tabaquismo activo y el del HDL. (3) Entrena un **Naïve Bayes** (TF-IDF + MultinomialNB) sobre el texto de `notas_clinicas.csv` para predecir la especialidad, con la exactitud y un par de predicciones de ejemplo. Compara la logística con un Random Forest y coméntame por qué aquí gana la simple."*

Tu trabajo con su respuesta es el de siempre: comprobar que la **partición** es honesta, que hay ***baseline***, que la **métrica** encaja con el coste del error y que en la logística se muestran los **odds ratio**. Ese criterio es lo que no se automatiza.
